# Inspect PFC First-Level Betas

Single-subject mean beta values within vmPFC for the two parametric modulator contrasts of interest.

Complements the unthresholded group T-map analysis: if the group mean T ≈ 1.3 in vmPFC is driven
by a consistent weak effect across subjects, the beta distribution should be right-shifted;
if it is driven by a few outliers, the distribution will show a heavy tail.

**Contrasts:**
- `con_0002.nii` → `first_stimxQval`
- `con_0003.nii` → `first_stimxHval`

**vmPFC mask:** Bartra et al. 2013, resampled to MNI152NLin2009cAsym.

GLM: `glm2_all_runs_scrubbed_2025-12-11-12-44`


In [ ]:
import os
import glob
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
from nilearn import image

GLM_DIR  = "/mnt/data/learning-habits/spm_format/outputs/glm2_all_runs_scrubbed_2025-12-11-12-44"
MASK_DIR = "/mnt/data/learning-habits/masks/MNI152NLin2009cAsym"

CONTRASTS = {
    "first_stimxQval": "con_0002.nii",
    "first_stimxHval": "con_0003.nii",
}

vmpfc_mask = nib.load(os.path.join(MASK_DIR, "vmpfc_bartra2013_MNI152NLin2009cAsym.nii"))
print(f"vmPFC mask voxels: {int((vmpfc_mask.get_fdata() > 0).sum())}")

sub_dirs = sorted(glob.glob(os.path.join(GLM_DIR, "sub-*")))
print(f"Found {len(sub_dirs)} subject directories")

betas = {name: [] for name in CONTRASTS}

for sub_dir in sub_dirs:
    subject = os.path.basename(sub_dir)
    for contrast_name, con_file in CONTRASTS.items():
        con_path = os.path.join(sub_dir, con_file)
        if not os.path.exists(con_path):
            print(f"  [SKIP] {subject} {contrast_name}: file not found")
            continue
        con_img  = nib.load(con_path)
        mask_res = image.resample_to_img(vmpfc_mask, con_img, interpolation='nearest')
        roi_mask = mask_res.get_fdata() > 0
        con_data = con_img.get_fdata()
        vals     = con_data[roi_mask]
        vals     = vals[np.isfinite(vals)]
        if len(vals) == 0:
            print(f"  [WARN] {subject} {contrast_name}: no finite voxels in mask")
            continue
        betas[contrast_name].append((subject, vals.mean()))

for name, data in betas.items():
    vals = np.array([v for _, v in data])
    print(f"{name}: n={len(data)}, mean={vals.mean():.4f}, sd={vals.std():.4f}")


In [ ]:
contrast_names = list(betas.keys())
fig, axes = plt.subplots(1, len(contrast_names), figsize=(6 * len(contrast_names), 5), sharey=True)

for ax, contrast_name in zip(axes, contrast_names):
    data  = betas[contrast_name]
    subs  = [s for s, _ in data]
    vals  = np.array([v for _, v in data])

    parts = ax.violinplot([vals], positions=[0], showmedians=True, showextrema=True)
    parts['bodies'][0].set_facecolor('#888888')
    parts['bodies'][0].set_alpha(0.5)
    parts['cmedians'].set_color('black')
    parts['cmedians'].set_linewidth(2)

    jitter = np.random.RandomState(42).uniform(-0.08, 0.08, len(vals))
    ax.scatter(jitter, vals, s=18, alpha=0.7, color='steelblue', zorder=3)

    ax.axhline(0, color='black', lw=1, ls='--', alpha=0.7)
    ax.set_xticks([0])
    ax.set_xticklabels([contrast_name])
    ax.set_ylabel('Mean beta within vmPFC')
    ax.set_title(f'{contrast_name}\n(n={len(vals)}, mean={vals.mean():.4f})', fontsize=11)
    ax.spines[['top', 'right']].set_visible(False)

plt.suptitle('First-level betas in vmPFC (Bartra 2013 mask)', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
print("Outlier subjects (|beta| > 2 SD from group mean):")
for contrast_name, data in betas.items():
    vals   = np.array([v for _, v in data])
    subs   = [s for s, _ in data]
    mean_b = vals.mean()
    std_b  = vals.std()
    outliers = [(s, v) for s, v in zip(subs, vals) if abs(v - mean_b) > 2 * std_b]
    if outliers:
        print(f"\n  {contrast_name}  (mean={mean_b:.4f}, sd={std_b:.4f})")
        for s, v in sorted(outliers, key=lambda x: abs(x[1] - mean_b), reverse=True):
            print(f"    {s}  beta={v:.4f}  ({(v - mean_b) / std_b:+.1f} SD)")
    else:
        print(f"\n  {contrast_name}: no outliers")


## Interpretation

*(Run notebook to populate)*

Expected pattern if there is a genuine positive Q-value effect in vmPFC:
- Beta distribution for `first_stimxQval` should be right-skewed (positive mean), consistent with mean T ≈ 1.3 seen in the unthresholded group map.

Expected pattern if vmPFC Q-value effect is absent:
- Distribution symmetric around zero; no consistent direction across subjects.

For `first_stimxHval`, the group T-map showed mean T ≈ −0.95 in vmPFC — the beta distribution should be slightly left-shifted if this is a real (weak) negative effect, or symmetric if it is noise.

Outlier subjects (|beta| > 2 SD) are flagged below the plot — these subjects disproportionately drive or suppress any group-level effect.
